# LangChain Transform Output Parsers Reference

Developer-facing statements defined in `langchain_core.output_parsers.transform`.

# `BaseTransformOutputParser: BaseOutputParser[T]`

Abstract base class for output parsers that transform streaming text or message inputs one chunk at a time.

A concrete subclass must implement the inherited `parse()` method from `BaseOutputParser`.

## Methods

### `transform`

Synchronously parses each input chunk through the runnable streaming infrastructure.

```python
@override
transform(
    self,
    input: Iterator[str | BaseMessage], # Stream of text or message chunks
    config: RunnableConfig | None = None, # Optional runnable configuration
    **kwargs: Any, # Additional accepted arguments
) -> Iterator[T] # Parsed output for each input chunk
```

Each `BaseMessage` input is wrapped in a single `ChatGeneration`. Each string input is wrapped in a single `Generation`. The resulting generation is passed to `parse_result()`.

The transformation runs through `_transform_stream_with_config()` using run type `"parser"`.

### `atransform`

Asynchronously parses each input chunk through the runnable streaming infrastructure.

```python
@override
async atransform(
    self,
    input: AsyncIterator[str | BaseMessage], # Asynchronous stream of text or message chunks
    config: RunnableConfig | None = None, # Optional runnable configuration
    **kwargs: Any, # Additional accepted arguments
) -> AsyncIterator[T] # Parsed output for each input chunk
```

Each `BaseMessage` input is wrapped in a single `ChatGeneration`. Each string input is wrapped in a single `Generation`.

`parse_result()` is executed through `run_in_executor()`. The transformation runs through `_atransform_stream_with_config()` using run type `"parser"`.

---

# `BaseCumulativeTransformOutputParser: BaseTransformOutputParser[T]`

Abstract base class for output parsers that accumulate streaming input and repeatedly parse the complete value received so far.

A concrete subclass must implement the inherited `parse()` method. Parsers that enable differential output must also provide the private difference calculation used by the cumulative transform; its default implementation raises `NotImplementedError`.

## Fields

```python
diff: bool = False # Whether streaming emits differences or complete cumulative parsed values
```

## Constructor

```python
BaseCumulativeTransformOutputParser(
    *,
    diff: bool = False, # Whether streaming emits differences instead of cumulative values
) -> None
```

## Behaviour

For every incoming string, the parser creates a `GenerationChunk`. A `BaseMessageChunk` becomes a `ChatGenerationChunk`. A complete `BaseMessage` is converted into a `BaseMessageChunk` from its model data and then wrapped in a `ChatGenerationChunk`.

Successive generation chunks are combined with the chunk addition operation. After each addition, the accumulated generation is passed to `parse_result(..., partial=True)`.

A parsed value is emitted only when it is not `None` and differs from the previously emitted parsed value.

When `diff=False`, the current cumulative parsed value is emitted.

When `diff=True`, the parser emits the result of its difference calculation between the previous parsed value and the current parsed value. The first difference receives `None` as the previous value.

The asynchronous cumulative transformation follows the same accumulation rules, calls `aparse_result(..., partial=True)`, and executes difference calculation through `run_in_executor()`.

In [ ]:
from collections.abc import AsyncIterator # Import the asynchronous iterator type

from langchain_core.messages import AIMessageChunk # Import a real LangChain streaming message chunk
from langchain_core.output_parsers.transform import BaseCumulativeTransformOutputParser, BaseTransformOutputParser # Import both streaming parser bases


class UppercaseTransformParser(BaseTransformOutputParser[str]): # Create a per-chunk transformation parser
    def parse(self, text: str) -> str: # Implement the required parse method
        return text.upper() # Convert each received chunk to uppercase


class CumulativeWordParser(BaseCumulativeTransformOutputParser[list[str]]): # Create a cumulative streaming parser
    def parse(self, text: str) -> list[str]: # Parse all text accumulated so far
        return text.split() # Return the accumulated text as a list of words

    def _diff( # Implement the difference calculation required when diff is enabled
        self,
        previous: list[str] | None, # Receive the previously emitted word list
        current: list[str], # Receive the current cumulative word list
    ) -> list[str]:
        previous_length = len(previous) if previous is not None else 0 # Count previously emitted words
        return current[previous_length:] # Return only newly added words


uppercase_parser = UppercaseTransformParser() # Create the per-chunk parser

text_chunks = iter([ # Create synchronous string chunks
    "LangChain ", # Provide the first chunk
    "supports ", # Provide the second chunk
    "streaming.", # Provide the final chunk
]) # Finish creating the iterator

print("Per-chunk string transformation:") # Display a heading

for parsed_chunk in uppercase_parser.transform(text_chunks): # Transform each chunk independently
    print(parsed_chunk) # Display the transformed chunk

message_chunks = iter([ # Create synchronous message chunks
    AIMessageChunk(content="hello "), # Provide the first message chunk
    AIMessageChunk(content="from "), # Provide the second message chunk
    AIMessageChunk(content="langchain"), # Provide the final message chunk
]) # Finish creating the message iterator

print("\nPer-chunk message transformation:") # Display a heading

for parsed_chunk in uppercase_parser.transform(message_chunks): # Transform each message chunk
    print(parsed_chunk) # Display the extracted uppercase text

cumulative_parser = CumulativeWordParser(diff=False) # Emit complete cumulative results

cumulative_chunks = iter([ # Create chunks ending at word boundaries
    "LangChain ", # Provide the first word
    "supports ", # Add the second word
    "streaming output.", # Add the remaining words
]) # Finish creating the iterator

print("\nCumulative results:") # Display a heading

for parsed_value in cumulative_parser.transform(cumulative_chunks): # Parse all accumulated text
    print(parsed_value) # Display each cumulative result

difference_parser = CumulativeWordParser(diff=True) # Emit only differences

difference_chunks = iter([ # Create another cumulative stream
    "Python ", # Provide the first word
    "and SQL ", # Add two more words
    "are useful.", # Add the final words
]) # Finish creating the iterator

print("\nDifference results:") # Display a heading

for parsed_difference in difference_parser.transform(difference_chunks): # Emit newly added words
    print(parsed_difference) # Display each difference


async def generate_async_chunks() -> AsyncIterator[str]: # Define an asynchronous text stream
    yield "Async " # Yield the first chunk
    yield "parsing " # Yield the second chunk
    yield "also works." # Yield the final chunk


async_parser = CumulativeWordParser(diff=False) # Create an asynchronous cumulative parser

print("\nAsynchronous cumulative results:") # Display a heading

async for parsed_value in async_parser.atransform(generate_async_chunks()): # Parse asynchronously in Jupyter
    print(parsed_value) # Display each cumulative result